In [ ]:
from __future__ import annotations
import pandas as pd, numpy as np
from pathlib import Path
from typing import Optional, Literal
from pandas.api.types import is_numeric_dtype, is_bool_dtype
import json, warnings

# Evidently
from evidently import Report
from evidently.presets import DataDriftPreset
from evidently.metrics import ValueDrift, DriftedColumnsCount
from evidently import DataDefinition, Dataset

warnings.filterwarnings("ignore")

In [16]:
plant_files = {
    "planta1": Path("../df_procesados/df_planta_1.csv"),
    "planta2": Path("../df_procesados/df_planta_2.csv"),
    "planta3": Path("../df_procesados/df_planta_3.csv"),
}

flag_files = {
    "planta1": Path("../df_procesados/flags_p1.csv"),
    "planta2": Path("../df_procesados/flags_p2.csv"),
    "planta3": Path("../df_procesados/flags_p3.csv")
}

ROOT_OUT = Path("../reportes_Drift")  # carpeta raíz; dentro se crea subcarpeta por planta
ROOT_OUT.mkdir(parents=True, exist_ok=True)

#---------------------------------------------------------------------------------#
# Ventanas por TIEMPO (no por número de filas)
# ── Ventanas
CURRENT_WINDOW: str = "24H"     # tramo "actual" si es por tiempo
CURRENT_BY_SAMPLES: bool = False
CURRENT_LAST_N: int = 1440      # si es por muestras (e.g., 1440 ≈ 24h a 1min)

# ── Baseline
BASELINE_STRATEGY: Literal["decay","golden","seasonal"] = "golden"  # default si no iteras
STRATEGIES_TO_RUN: list[Literal["decay","golden","seasonal"]] = ["golden","decay","seasonal"]

DECAY_HALF_LIFE_HOURS: int = 24*7
DECAY_WEIGHT_MASS: float = 0.95

GOLDEN_WIN: str = "30min"
GOLDEN_STEP: str = "10min"
GOLDEN_K: int = 40

SEASONAL_WEEKS_BACK: int = 12
# ── Resample (opcional)
RESAMPLE: Optional[str] = None
RESAMPLE_AGG: Literal["mean","median"] = "mean"

# ── Exclusiones y KPI
EXCLUDE_COLUMNS: list[str] = [
    'pH Ecualizador 2 (Tk 250m3)', "Conductividad DAF", "Temperatura DAF",
    'pH entrada a Ecualizador 1', "Flujo Aire Reactor 1", "OD Reactor 1"
]
KPI_ENABLED = False
KPI_COL     = ""
KPI_IN_FILENAME = True

# ── Evidently
NUM_METHOD: Literal["auto","ks","wasserstein","psi","anderson","cramer","mannwhitney"] = "auto"
NUM_THRESHOLD: Optional[float] = None

# ── Salidas tabulares
SAVE_CSV  = False  # si True, guarda per-column y resumen por corrida

In [17]:
def strip_outliers(df: pd.DataFrame) -> pd.DataFrame:
    if "is_outlier" not in df.columns:
        return df
    s = df["is_outlier"]
    mask = ~(s.astype(str).str.lower().isin(["1","true","t","yes","y"]))
    return df.loc[mask].drop(columns=["is_outlier"])

def resample_mixed(df: pd.DataFrame, freq: str, agg: str) -> pd.DataFrame:
    """Resample mixto: numéricas por mean/median; categóricas por moda."""
    if df.empty: return df
    num = df.select_dtypes(include="number")
    other_cols = [c for c in df.columns if c not in num.columns]
    num_rs = num.resample(freq).median() if agg == "median" else num.resample(freq).mean()
    if other_cols:
        def _mode(s: pd.Series):
            s = s.dropna()
            if s.empty: return np.nan
            counts = s.value_counts()
            return counts.index[0]
        other = df[other_cols]
        other_rs = other.resample(freq).agg(_mode)
        out = pd.concat([num_rs, other_rs], axis=1)
    else:
        out = num_rs
    out = out[[c for c in df.columns if c in out.columns]]
    return out

def build_types_keep_all(ref: pd.DataFrame, cur: pd.DataFrame, dt_col: str) -> tuple[list[str], list[str], list[str]]:
    """Devuelve (numeric_cols, categorical_cols, dropped_all_nan)."""
    common = [c for c in ref.columns.intersection(cur.columns) if c != dt_col and c not in EXCLUDE_COLUMNS]
    numeric_cols, categorical_cols, dropped_all_nan = [], [], []
    for c in common:
        r, k = ref[c], cur[c]
        if r.dropna().empty and k.dropna().empty:
            dropped_all_nan.append(c); continue
        if is_bool_dtype(r) or is_bool_dtype(k):
            categorical_cols.append(c)
        elif is_numeric_dtype(r) or is_numeric_dtype(k):
            numeric_cols.append(c)
        else:
            categorical_cols.append(c)
    return numeric_cols, categorical_cols, dropped_all_nan

def window_starts(index: pd.DatetimeIndex, win: pd.Timedelta, step: pd.Timedelta):
    if len(index) == 0: return []
    t, tmax = index.min(), index.max()
    out = []
    while t + win <= tmax:
        out.append(t); t = t + step
    return out


In [18]:
def ref_decay_prefix_mass(df_hist: pd.DataFrame, now: pd.Timestamp,
                          half_life_hours=24*7, target_mass=0.95) -> pd.DataFrame:
    """Pesos w = exp(-Δt/τ), toma prefijo más reciente con masa acumulada >= target_mass."""
    if df_hist.empty: return df_hist
    tau = pd.Timedelta(hours=half_life_hours) / np.log(2)
    dt = (now - df_hist.index)
    # convertir Timedelta a float para exponencial
    w = np.exp(-(dt / tau).astype(float))
    order = np.argsort(-df_hist.index.asi8)  # descendente por tiempo
    w_sorted = w.values[order]
    cum = np.cumsum(w_sorted) / w_sorted.sum()
    cut_idx = np.searchsorted(cum, target_mass, side="left")
    take_pos = order[: (cut_idx + 1)]
    sel = df_hist.iloc[np.sort(take_pos)]
    return sel

def ref_golden(df_hist: pd.DataFrame, win="30min", step="10min", k=40) -> pd.DataFrame:
    """Elige K ventanas históricas más 'estables' (score robusto)."""
    win_td, step_td = pd.to_timedelta(win), pd.to_timedelta(step)
    starts = window_starts(df_hist.index, win_td, step_td)
    if not starts: return df_hist.iloc[:0]
    rows = []
    for t0 in starts:
        t1 = t0 + win_td - pd.Timedelta(nanoseconds=1)
        sub = df_hist.loc[t0:t1]
        if len(sub) < 3: continue
        num = sub.select_dtypes(include="number")
        if num.shape[1] == 0: continue
        med = num.median()
        iqr = num.quantile(0.75) - num.quantile(0.25)
        rsd = (iqr / (med.abs() + 1e-12)).replace([np.inf, -np.inf], np.nan)
        score = rsd.median(skipna=True)
        rows.append((t0, t1, float(score)))
    if not rows: return df_hist.iloc[:0]
    stab = pd.DataFrame(rows, columns=["t0","t1","score"]).sort_values("score").head(k)
    parts = [df_hist.loc[t0:t1] for t0, t1, _ in stab.itertuples(index=False)]
    return pd.concat(parts, axis=0) if parts else df_hist.iloc[:0]

def ref_seasonal(df_hist: pd.DataFrame, current_end: pd.Timestamp, weeks_back=12) -> pd.DataFrame:
    """Mismo día-de-semana y hora-del-día, W semanas atrás."""
    if df_hist.empty: return df_hist.iloc[:0]
    slot = current_end.dayofweek * 24 + current_end.hour
    dw, hh = df_hist.index.dayofweek, df_hist.index.hour
    mask = (dw * 24 + hh) == slot
    hist = df_hist.loc[mask].loc[:current_end]
    if hist.empty: return df_hist.iloc[:0]
    start_lim = current_end - pd.Timedelta(weeks=weeks_back)
    return hist.loc[start_lim:]

In [19]:
def extract_value_drift_table(report, snap=None) -> pd.DataFrame:
    """Tabla por columna: col, drifted, score, method, threshold."""
    d = None

    # 1) Intentar con Report
    if hasattr(report, "as_dict"):
        d = report.as_dict()
    elif hasattr(report, "json"):
        j = report.json()
        d = json.loads(j) if isinstance(j, str) else j

    # 2) Si no se pudo con Report, intentar con Snapshot
    if d is None and snap is not None:
        if hasattr(snap, "as_dict"):
            d = snap.as_dict()
        elif hasattr(snap, "json"):
            j = snap.json()
            d = json.loads(j) if isinstance(j, str) else j

    if d is None:
        raise RuntimeError("No se pudo serializar Report/Snapshot a dict/json.")

    rows = []
    for m in d.get("metrics", []):
        if m.get("metric") == "ValueDrift":
            res = m.get("result", {}) or {}
            rows.append({
                "col":       res.get("column_name") or res.get("column"),
                "drifted":   res.get("drift_detected"),
                "score":     res.get("drift_score"),
                "method":    res.get("stattest_name") or res.get("stattest"),
                "threshold": res.get("drift_threshold") or res.get("threshold"),
            })
    return pd.DataFrame(rows)


In [20]:
def make_report_for_plant(
    df: pd.DataFrame,
    out_dir: Path,
    strategy: Literal["decay","golden","seasonal"] = BASELINE_STRATEGY,
    forced_dt_col: Optional[str] = "date_time",
    out_prefix: str = "planta",
) -> Path:
    # 1) índice temporal + outliers
    dt = "date_time"
    df = df.copy()
    df[dt] = pd.to_datetime(df[dt], errors="coerce")
    df = df.dropna(subset=[dt]).sort_values(dt).set_index(dt)
    df = strip_outliers(df)
    if df.empty:
        raise ValueError("Dataset vacío tras filtrar/parsear fechas.")

    # 2) flags por columna (si existen)
    flag_path = flag_files.get(out_prefix)
    if flag_path and flag_path.exists():
        flags = pd.read_csv(flag_path, parse_dates=["date_time"])
        flags["date_time"] = pd.to_datetime(flags["date_time"]).dt.floor("min")
        df.index = df.index.floor("min")
        df = df.merge(flags, left_index=True, right_on="date_time", how="left").set_index("date_time")

        nd_cols = [c for c in df.columns if c.startswith("nd_")]
        for nd_col in nd_cols:
            var = nd_col.replace("nd_", "")
            if var in df.columns:
                # si tu semántica de nd_ es "no dato", usa valid_mask = ~df[nd_col]
                valid_mask = ~df[nd_col]
                df.loc[~valid_mask, var] = np.nan

        drop_cols = ["valid_for_drift", "nd_any", "nd_all"] + nd_cols
        df = df[[c for c in df.columns if c not in drop_cols]]
        print(f"[{out_prefix}] Flags integrados por columna → {len(df)} filas.")
    else:
        print(f"[{out_prefix}] Sin flags o archivo no encontrado, se usa DF completo.")

    # 3) split temporal (por tiempo o por muestras)
    now = df.index.max()
    if CURRENT_BY_SAMPLES:
        cur = df.tail(CURRENT_LAST_N)
        cur_start = cur.index.min()
        hist = df.loc[:cur_start - pd.Timedelta(nanoseconds=1)]
    else:
        cur_start = now - pd.to_timedelta(CURRENT_WINDOW)
        cur = df.loc[cur_start:now]
        hist = df.loc[:cur_start - pd.Timedelta(nanoseconds=1)]

    # 4) baseline determinística
    if strategy == "decay":
        ref_global = ref_decay_prefix_mass(hist, now, DECAY_HALF_LIFE_HOURS, DECAY_WEIGHT_MASS)
    elif strategy == "golden":
        ref_global = ref_golden(hist, GOLDEN_WIN, GOLDEN_STEP, GOLDEN_K)
    elif strategy == "seasonal":
        ref_global = ref_seasonal(hist, now, SEASONAL_WEEKS_BACK)
    else:
        raise ValueError("strategy inválida")
    if ref_global.empty:
        ref_global = hist

    # 5) columnas comunes (sin excluidas)
    common_cols = sorted(set(ref_global.columns).intersection(cur.columns) - set(EXCLUDE_COLUMNS))
    if not common_cols:
        raise ValueError("No hay columnas comunes para comparar.")
    ref_final = ref_global[common_cols].copy()
    cur_final = cur[common_cols].copy()

    # 6) resample (opcional)
    if RESAMPLE:
        ref_final = resample_mixed(ref_final, RESAMPLE, RESAMPLE_AGG).dropna(how="all")
        cur_final = resample_mixed(cur_final, RESAMPLE, RESAMPLE_AGG).dropna(how="all")

    # 7) tipos Evidently
    numeric_cols, categorical_cols, dropped_all_nan = build_types_keep_all(ref_final, cur_final, dt_col=dt)
    if not numeric_cols and not categorical_cols:
        raise ValueError("Todas las columnas quedaron 100% NaN en ref y cur tras el resample.")

    definition = DataDefinition(
        numerical_columns=numeric_cols if numeric_cols else None,
        categorical_columns=categorical_cols if categorical_cols else None
    )

    preset_kwargs = {}
    if NUM_METHOD != "auto":
        preset_kwargs["num_method"] = NUM_METHOD
        if NUM_THRESHOLD is not None:
            preset_kwargs["num_threshold"] = NUM_THRESHOLD

    metrics = [
        DataDriftPreset(**preset_kwargs),
        DriftedColumnsCount(**preset_kwargs),
        *[
            (ValueDrift(column=c) if NUM_METHOD == "auto"
             else (ValueDrift(column=c, method=NUM_METHOD) if NUM_THRESHOLD is None
                   else ValueDrift(column=c, method=NUM_METHOD, threshold=NUM_THRESHOLD)))
            for c in (numeric_cols + categorical_cols)
        ],
    ]

    # 8) Evidently run
    report = Report(metrics=metrics)
    ds_ref = Dataset.from_pandas(ref_final.reset_index(drop=True), data_definition=definition)
    ds_cur = Dataset.from_pandas(cur_final.reset_index(drop=True), data_definition=definition)


    snap = report.run(reference_data=ds_ref, current_data=ds_cur)

    # 9) extracción & guardado
    df_cols = extract_value_drift_table(report, snap)   # ← UNA sola llamada, con snap
    drifted_count = int(df_cols.get("drifted", pd.Series(dtype=bool)).fillna(False).sum())
    total_cols = int(df_cols.shape[0])

    kpi_tag = ""
    kpi_present = KPI_ENABLED and (KPI_COL in df_cols["col"].tolist())
    kpi_drifted = bool(df_cols.query("col == @KPI_COL and drifted == True").shape[0]) if kpi_present else False
    if KPI_ENABLED and KPI_IN_FILENAME:
        kpi_tag = "_KPI-DRIFT" if kpi_drifted else "_KPI-OK" if kpi_present else "_KPI-N/A"

    out_html = out_dir / f"{out_prefix}_{strategy}.html"

    # extrae la tabla una sola vez, usando snap
    df_cols = extract_value_drift_table(report, snap)

    # guardar HTML con tolerancia de versión
    if hasattr(snap, "save_html"):
        snap.save_html(str(out_html))
    elif hasattr(snap, "as_html"):
        out_html.write_text(snap.as_html(), encoding="utf-8")
    elif hasattr(report, "save_html"):
        report.save_html(str(out_html))
    elif hasattr(report, "as_html"):
        out_html.write_text(report.as_html(), encoding="utf-8")
    else:
        raise RuntimeError("Tu Evidently no expone save_html/as_html en Report ni Snapshot.")
    if SAVE_CSV:
        tag_base = f"{strategy}_{'resamp'+RESAMPLE if RESAMPLE else 'raw'}_{pd.Timestamp(now).strftime('%Y%m%d_%H%M%S')}"
        df_cols.to_csv(out_dir / f"{out_prefix}_per_column_{tag_base}{kpi_tag}.csv", index=False)
        run_summary = pd.DataFrame([{
            "plant": out_prefix,
            "run_time": pd.Timestamp(now),
            "strategy": strategy,
            "current_window": (f"last_{CURRENT_LAST_N}_rows" if CURRENT_BY_SAMPLES else CURRENT_WINDOW),
            "resample": RESAMPLE or "raw",
            "resample_agg": RESAMPLE_AGG,
            "num_method": NUM_METHOD,
            "drifted_cols": drifted_count,
            "total_cols": total_cols,
            "kpi_present": kpi_present,
            "kpi_col": KPI_COL if kpi_present else None,
            "kpi_drifted": kpi_drifted,
            "html_file": out_html.name,
        }])
        timeline_path = out_dir / "timeline_runs.csv"
        path_summary = out_dir / f"{out_prefix}_run_summary_{tag_base}{kpi_tag}.csv"
        run_summary.to_csv(path_summary, index=False)
        header = not timeline_path.exists()
        run_summary.to_csv(timeline_path, mode="a", header=header, index=False)

    print(f"[{out_prefix}] cols={total_cols} | drifted={drifted_count} | KPI={'DRIFT' if kpi_drifted else 'OK' if kpi_present else 'N/A'}")
    print(f"OK → {out_html.name} (carpeta: {out_dir})")
    return out_html

In [21]:
for plant, csv_path in plant_files.items():
    # subcarpeta por planta
    plant_out = ROOT_OUT / plant
    plant_out.mkdir(parents=True, exist_ok=True)

    df = pd.read_csv(csv_path)

    # doble loop: estrategias por planta
    for strategy in STRATEGIES_TO_RUN:
        try:
            make_report_for_plant(
                df=df,
                out_dir=plant_out,
                strategy=strategy,
                out_prefix=plant
            )
        except Exception as e:
            print(f"[{plant}][{strategy}] ERROR: {e}")

[planta1] Flags integrados por columna → 82286 filas.
[planta1] cols=0 | drifted=0 | KPI=N/A
OK → planta1_golden.html (carpeta: ..\reportes_Drift\planta1)
[planta1] Flags integrados por columna → 82286 filas.
[planta1] cols=0 | drifted=0 | KPI=N/A
OK → planta1_decay.html (carpeta: ..\reportes_Drift\planta1)
[planta1] Flags integrados por columna → 82286 filas.
[planta1] cols=0 | drifted=0 | KPI=N/A
OK → planta1_seasonal.html (carpeta: ..\reportes_Drift\planta1)
[planta2] Flags integrados por columna → 440471 filas.
[planta2] cols=0 | drifted=0 | KPI=N/A
OK → planta2_golden.html (carpeta: ..\reportes_Drift\planta2)
[planta2] Flags integrados por columna → 440471 filas.
[planta2] cols=0 | drifted=0 | KPI=N/A
OK → planta2_decay.html (carpeta: ..\reportes_Drift\planta2)
[planta2] Flags integrados por columna → 440471 filas.
[planta2] cols=0 | drifted=0 | KPI=N/A
OK → planta2_seasonal.html (carpeta: ..\reportes_Drift\planta2)
[planta3] Flags integrados por columna → 22568 filas.
[planta3] c